# 01 · Training data exploration

I investigate data integrity, subgroup composition, length, potential leakage, and within-document variation before selecting a validation design. I use only the provided training data, no classifier, and no test data. Descriptive associations are not evidence of causation or model performance. All sampling uses seed 2026.

In [1]:
from pathlib import Path
import hashlib
import json
import re
import unicodedata
from collections import Counter, defaultdict
from difflib import SequenceMatcher

import numpy as np
import plotly.express as px
import polars as pl
from IPython.display import Markdown, display
from huggingface_hub import hf_hub_download
from tokenizers import Tokenizer

ROOT = Path.cwd() if (Path.cwd() / "pyproject.toml").exists() else Path.cwd().parent
TRAIN_PATH = ROOT / "data" / "train.csv"
METADATA_PATH = ROOT / "data" / "metadata" / "eda_summary.json"
SEED = 2026
LABELS = ["human", "ai", "fusion"]
COLORS = {"human": "#2878a5", "ai": "#e07b39", "fusion": "#7c5da8"}
train = pl.read_csv(TRAIN_PATH, infer_schema_length=10000)
summary: dict[str, object] = {
    "seed": SEED,
    "input_sha256": hashlib.file_digest(TRAIN_PATH.open("rb"), "sha256").hexdigest(),
    "rows": train.height,
    "columns": train.columns,
    "schema": {k: str(v) for k, v in train.schema.items()},
}
summary

{'seed': 2026,
 'input_sha256': 'd55823815042eed2937e92e1018f5595915b8f7573d861048956521434f65025',
 'rows': 272352,
 'columns': ['id', 'text', 'lang', 'domain', 'label'],
 'schema': {'id': 'String',
  'text': 'String',
  'lang': 'String',
  'domain': 'String',
  'label': 'String'}}

## 01 · Integrity and conservative duplicates

**Question.** Are records complete and structurally consistent? I distinguish exact matches from matches after Unicode NFKC and whitespace normalization. NFKC can merge legitimate typography, so normalized matches are candidates, not necessarily errors. Replacement characters and control characters are indicators, not proof of encoding corruption.

In [2]:
def normalize_text(text: str) -> str:
    return " ".join(unicodedata.normalize("NFKC", text).split())


quality = {f"null_{name}": train[name].null_count() for name in train.columns}
quality.update(
    {
        "empty_text": train.filter(pl.col("text") == "").height,
        "whitespace_only_text": train.filter(
            (pl.col("text").str.strip_chars() == "") & (pl.col("text") != "")
        ).height,
        "duplicate_id_excess": train.height - train["id"].n_unique(),
        "duplicate_row_excess": train.height - train.n_unique(),
        "replacement_character_rows": train.filter(
            pl.col("text").str.contains("�", literal=True)
        ).height,
        "control_character_rows": train.filter(
            pl.col("text").str.contains(r"[\x00-\x08\x0b\x0c\x0e-\x1f]")
        ).height,
    }
)
train = train.with_columns(
    pl.col("text").map_elements(normalize_text, return_dtype=pl.String).alias("normalized_text")
)
quality["duplicate_raw_text_excess"] = train.height - train["text"].n_unique()
quality["duplicate_normalized_text_excess"] = train.height - train["normalized_text"].n_unique()
quality["unexpected_label"] = train.filter(~pl.col("label").is_in(LABELS)).height
quality["unexpected_language"] = train.filter(~pl.col("lang").is_in(["en-EN", "de-DE"])).height
summary["quality"] = quality
quality_table = pl.DataFrame(
    {"issue": list(quality), "count": list(quality.values())}
).with_columns((100 * pl.col("count") / train.height).round(3).alias("percent"))
display(quality_table)
display(train.select("label", "lang", "domain").unique().sort(["lang", "domain", "label"]))

issue,count,percent
str,i64,f64
"""null_id""",0,0.0
"""null_text""",0,0.0
"""null_lang""",0,0.0
"""null_domain""",0,0.0
"""null_label""",0,0.0
…,…,…
"""control_character_rows""",0,0.0
"""duplicate_raw_text_excess""",0,0.0
"""duplicate_normalized_text_exce…",1,0.0


label,lang,domain
str,str,str
"""ai""","""de-DE""","""bundestag"""
"""fusion""","""de-DE""","""bundestag"""
"""human""","""de-DE""","""bundestag"""
"""ai""","""de-DE""","""gutenberg"""
"""fusion""","""de-DE""","""gutenberg"""
…,…,…
"""fusion""","""en-EN""","""house_of_commons"""
"""human""","""en-EN""","""house_of_commons"""
"""ai""","""en-EN""","""student_essays"""


**Measured observations.** The 272,352 rows have no nulls, duplicate IDs, exact duplicate rows, empty texts, or unexpected class and language values. There are 34 rows containing a Unicode replacement character. One pair becomes identical after NFKC and whitespace normalization; the raw texts are distinct. The replacement character may be legitimate source content, so I do not infer an encoding failure.

**Interpretation.** Null counts and duplicate excess counts have different meanings: excess counts count records beyond the first in each group. The observed domain values below define coverage; domain spelling is not treated as invalid without an authoritative vocabulary.

## 02–03 · Class, language, and domain

**Question.** Could metadata or uneven subgroup coverage create shortcuts or unstable macro-F1? I compare counts and percentages within each language and domain. Percentages describe associations, not evidence that a model will use metadata.

In [3]:
class_counts = (
    train.group_by("label")
    .len()
    .sort("label")
    .with_columns((100 * pl.col("len") / train.height).round(2).alias("percent"))
)
groups = (
    train.group_by(["lang", "domain", "label"])
    .len()
    .sort(["lang", "domain", "label"])
    .with_columns(
        (100 * pl.col("len") / pl.col("len").sum().over(["lang", "domain"]))
        .round(2)
        .alias("within_group_percent")
    )
)
summary["class_counts"] = class_counts.to_dicts()
summary["groups"] = groups.to_dicts()
language_counts = train.group_by("lang").len().sort("lang")
domain_counts = train.group_by("domain").len().sort("domain")
summary["language_counts"] = language_counts.to_dicts()
summary["domain_counts"] = domain_counts.to_dicts()
display(
    class_counts,
    language_counts,
    domain_counts,
    groups,
)
for group in ["lang", "domain"]:
    counts = (
        train.group_by([group, "label"])
        .len()
        .with_columns(
            (100 * pl.col("len") / pl.col("len").sum().over(group)).alias("Within group percent")
        )
    )
    display(
        px.bar(
            counts,
            x=group,
            y="Within group percent",
            color="label",
            barmode="stack",
            color_discrete_map=COLORS,
            title=f"Class distribution within {group}",
            labels={group: group.capitalize(), "label": "Class"},
        )
    )
display(
    px.density_heatmap(
        groups,
        x="domain",
        y="lang",
        z="len",
        histfunc="sum",
        title="Training examples by language and domain",
        labels={"domain": "Domain", "lang": "Language", "len": "Examples"},
    )
)

label,len,percent
str,u32,f64
"""ai""",121940,44.77
"""fusion""",83682,30.73
"""human""",66730,24.5


lang,len
str,i64
"""de-DE""",52771
"""en-EN""",219581


domain,len
str,i64
"""arxiv_papers""",14290
"""blog_authorship_corpus""",40547
"""bundestag""",52343
"""cnn_news""",44285
"""euro_court_cases""",27578
"""gutenberg""",7344
"""house_of_commons""",31859
"""student_essays""",54106


lang,domain,label,len,within_group_percent
str,str,str,u32,f64
"""de-DE""","""bundestag""","""ai""",19977,38.17
"""de-DE""","""bundestag""","""fusion""",17887,34.17
"""de-DE""","""bundestag""","""human""",14479,27.66
"""de-DE""","""gutenberg""","""ai""",360,84.11
"""de-DE""","""gutenberg""","""fusion""",37,8.64
…,…,…,…,…
"""en-EN""","""house_of_commons""","""fusion""",9608,30.16
"""en-EN""","""house_of_commons""","""human""",6953,21.82
"""en-EN""","""student_essays""","""ai""",21583,39.89


In [4]:
for field, title in [("lang", "Language coverage"), ("domain", "Domain coverage")]:
    counts = train.group_by(field).len().sort("len", descending=True)
    display(
        px.bar(
            counts,
            x=field,
            y="len",
            title=title,
            labels={field: field.capitalize(), "len": "Training examples"},
        )
    )
for field in ["lang", "domain"]:
    counts = train.group_by([field, "label"]).len()
    display(
        px.density_heatmap(
            counts,
            x=field,
            y="label",
            z="len",
            histfunc="sum",
            title=f"Class by {field}: training examples",
            labels={field: field.capitalize(), "label": "Class", "len": "Examples"},
        )
    )

**Measured observations.** The classes are 44.77% `ai`, 30.73% `fusion`, and 24.50% `human`. The `arxiv_papers` group is 84.32% `ai` and 2.85% `human`; English `gutenberg` is 91.95% `ai` and 3.12% `human`. These proportions create a plausible domain shortcut; they do not establish that a model uses it. German `gutenberg` contains only 31 human and 37 fusion examples.

Subgroup counts should inform representative sampling and subgroup uncertainty. Macro-F1 weights each class equally even when class support differs; inspect class-level results as well as the aggregate.

## 04 · Document length

**Question.** Do document sizes vary with class and subgroup? I count words as whitespace-delimited units, approximate sentence boundaries by punctuation followed by whitespace, and paragraphs by blank-line separators. These are proxies rather than linguistic annotation.

In [5]:
train = train.with_columns(
    pl.col("text").str.len_chars().alias("characters"),
    pl.col("text").str.count_matches(r"\S+").alias("words"),
    (pl.col("text").str.count_matches(r"[.!?]+(?:\s|$)") + 1).alias("sentences_proxy"),
    (pl.col("text").str.count_matches(r"\n\s*\n") + 1).alias("paragraphs_proxy"),
    pl.col("text").str.count_matches("\n").alias("newlines"),
    pl.col("text").str.count_matches(r"\S").alias("non_whitespace"),
).with_columns(
    (pl.col("non_whitespace") / pl.col("words").clip(lower_bound=1)).alias("characters_per_word"),
    (pl.col("words") / pl.col("sentences_proxy")).alias("words_per_sentence_proxy"),
)
length_columns = [
    "characters",
    "words",
    "sentences_proxy",
    "paragraphs_proxy",
    "newlines",
    "non_whitespace",
    "characters_per_word",
    "words_per_sentence_proxy",
]


def describe_lengths(data: pl.DataFrame, group: list[str]) -> pl.DataFrame:
    expressions = []
    for name in length_columns:
        for stat, expression in [
            ("mean", pl.col(name).mean()),
            ("std", pl.col(name).std()),
            ("min", pl.col(name).min()),
            ("p05", pl.col(name).quantile(0.05)),
            ("p25", pl.col(name).quantile(0.25)),
            ("median", pl.col(name).median()),
            ("p75", pl.col(name).quantile(0.75)),
            ("p90", pl.col(name).quantile(0.90)),
            ("p95", pl.col(name).quantile(0.95)),
            ("p99", pl.col(name).quantile(0.99)),
            ("max", pl.col(name).max()),
        ]:
            expressions.append(expression.alias(f"{name}_{stat}"))
    return data.group_by(group).agg(expressions).sort(group) if group else data.select(expressions)


length_overall = describe_lengths(train, [])
length_by_label = describe_lengths(train, ["label"])
summary["length_overall"] = length_overall.to_dicts()
summary["length_by_label"] = length_by_label.to_dicts()
display(
    length_overall,
    length_by_label.select(
        "label",
        *[
            f"words_{s}"
            for s in [
                "mean",
                "std",
                "min",
                "p05",
                "p25",
                "median",
                "p75",
                "p90",
                "p95",
                "p99",
                "max",
            ]
        ],
    ),
)
for grouping in [["lang"], ["domain"], ["lang", "label"]]:
    display(
        describe_lengths(train, grouping).select(
            *grouping, "words_median", "words_p95", "characters_median"
        )
    )
plot_sample = train.select("label", "lang", "words").sample(n=min(15000, train.height), seed=SEED)
display(
    px.box(
        plot_sample,
        x="label",
        y="words",
        color="label",
        facet_col="lang",
        log_y=True,
        points=False,
        color_discrete_map=COLORS,
        title="Word count by class and language",
        labels={"label": "Class", "words": "Words (log scale)", "lang": "Language"},
    )
)

characters_mean,characters_std,characters_min,characters_p05,characters_p25,characters_median,characters_p75,characters_p90,characters_p95,characters_p99,characters_max,words_mean,words_std,words_min,words_p05,words_p25,words_median,words_p75,words_p90,words_p95,words_p99,words_max,sentences_proxy_mean,sentences_proxy_std,sentences_proxy_min,sentences_proxy_p05,sentences_proxy_p25,sentences_proxy_median,sentences_proxy_p75,sentences_proxy_p90,sentences_proxy_p95,sentences_proxy_p99,sentences_proxy_max,paragraphs_proxy_mean,paragraphs_proxy_std,paragraphs_proxy_min,paragraphs_proxy_p05,…,newlines_p90,newlines_p95,newlines_p99,newlines_max,non_whitespace_mean,non_whitespace_std,non_whitespace_min,non_whitespace_p05,non_whitespace_p25,non_whitespace_median,non_whitespace_p75,non_whitespace_p90,non_whitespace_p95,non_whitespace_p99,non_whitespace_max,characters_per_word_mean,characters_per_word_std,characters_per_word_min,characters_per_word_p05,characters_per_word_p25,characters_per_word_median,characters_per_word_p75,characters_per_word_p90,characters_per_word_p95,characters_per_word_p99,characters_per_word_max,words_per_sentence_proxy_mean,words_per_sentence_proxy_std,words_per_sentence_proxy_min,words_per_sentence_proxy_p05,words_per_sentence_proxy_p25,words_per_sentence_proxy_median,words_per_sentence_proxy_p75,words_per_sentence_proxy_p90,words_per_sentence_proxy_p95,words_per_sentence_proxy_p99,words_per_sentence_proxy_max
f64,f64,u32,f64,f64,f64,f64,f64,f64,f64,u32,f64,f64,u32,f64,f64,f64,f64,f64,f64,f64,u32,f64,f64,u32,f64,f64,f64,f64,f64,f64,f64,u32,f64,f64,u32,f64,…,f64,f64,f64,u32,f64,f64,u32,f64,f64,f64,f64,f64,f64,f64,u32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
3444.098912,2679.665132,76,381.0,1631.0,2977.0,4487.0,6318.0,8074.0,14798.0,34876,542.336436,426.749048,20,65.0,273.0,463.0,686.0,974.0,1263.0,2450.0,3000,32.14124,27.564563,1,5.0,16.0,27.0,39.0,58.0,78.0,139.0,2374,1.0,0.0,1,1.0,…,0.0,0.0,0.0,0,2902.762477,2258.803971,54,316.0,1356.0,2511.0,3799.0,5352.0,6826.0,12359.0,32566,5.290977,0.691837,2.347826,4.254902,4.779927,5.204943,5.821121,6.216216,6.429412,6.811258,27.8,17.325971,7.731283,1.079051,9.526316,13.909091,16.769231,19.948718,23.1,25.354839,34.692308,657.0


label,words_mean,words_std,words_min,words_p05,words_p25,words_median,words_p75,words_p90,words_p95,words_p99,words_max
str,f64,f64,u32,f64,f64,f64,f64,f64,f64,f64,u32
"""ai""",493.478178,294.317882,20,60.0,287.0,465.0,657.0,872.0,1008.0,1307.0,2996
"""fusion""",585.151657,530.533505,20,79.0,263.0,445.0,695.0,1137.0,1820.0,2747.0,3000
"""human""",577.926315,473.39346,20,57.0,262.0,481.0,740.0,1105.0,1490.0,2533.0,2998


lang,words_median,words_p95,characters_median
str,f64,f64,f64
"""de-DE""",512.0,965.0,3631.0
"""en-EN""",444.0,1369.0,2705.0


domain,words_median,words_p95,characters_median
str,f64,f64,f64
"""arxiv_papers""",888.0,2632.0,6397.5
"""blog_authorship_corpus""",131.0,627.0,732.0
"""bundestag""",512.0,964.0,3636.0
"""cnn_news""",548.0,1133.0,3392.0
"""euro_court_cases""",633.0,2227.0,4173.0
"""gutenberg""",706.0,1738.0,4511.5
"""house_of_commons""",540.0,1760.0,3463.0
"""student_essays""",363.0,715.0,2134.0


lang,label,words_median,words_p95,characters_median
str,str,f64,f64,f64
"""de-DE""","""ai""",430.0,710.0,3106.0
"""de-DE""","""fusion""",584.0,1048.0,4098.5
"""de-DE""","""human""",584.0,1075.0,4059.5
"""en-EN""","""ai""",481.0,1040.0,3115.0
"""en-EN""","""fusion""",389.0,2032.0,2313.0
"""en-EN""","""human""",439.0,1618.0,2498.5


**Measured observations.** The median document has 463 words. The 95th percentile is 1,263 words. Median word counts are 465 for `ai`, 445 for `fusion`, and 481 for `human`; the respective 95th percentiles are 1,008, 1,820, and 1,490. These distributions overlap, while their tails differ. All texts contain zero newline characters, so paragraph boundaries cannot be recovered from this representation; a one-paragraph proxy does not mean the original documents had one paragraph.

Differences in marginal lengths may reflect domain and language composition. I do not infer a causal class effect or a model shortcut from these summaries alone.

## 05 · XLM-R tokenization and truncation

**Question.** How often does the input exceed practical token budgets? I sample up to 100 examples per observed language × domain × label group using a fixed seed. This intentionally oversamples rare groups; aggregate estimates are reweighted by their full training-group sizes. Token counts include XLM-R special tokens and are not word counts. Only the tokenizer is downloaded; model weights are not loaded.

In [6]:
sample = (
    train.group_by(["lang", "domain", "label"], maintain_order=True)
    .map_groups(lambda group: group.sample(n=min(100, group.height), seed=SEED))
    .sort(["lang", "domain", "label", "id"])
)
tokenizer_path = hf_hub_download("FacebookAI/xlm-roberta-base", "tokenizer.json")
tokenizer = Tokenizer.from_file(tokenizer_path)
token_lengths = []
for start in range(0, sample.height, 128):
    token_lengths.extend(
        len(item.ids)
        for item in tokenizer.encode_batch(sample["text"][start : start + 128].to_list())
    )
sample = sample.with_columns(pl.Series("tokens", token_lengths))
token_summary = sample.group_by(["lang", "domain", "label"]).agg(
    pl.len().alias("sample_size"),
    pl.col("tokens").median().alias("median_tokens"),
    pl.col("tokens").mean().alias("mean_tokens"),
    *[(pl.col("tokens") > limit).mean().alias(f"over_{limit}") for limit in [128, 256, 512, 1024]],
    ((pl.col("tokens") - 512).clip(lower_bound=0) / pl.col("tokens"))
    .mean()
    .alias("mean_discarded_fraction_512"),
)
token_weighted = token_summary.join(
    groups.select("lang", "domain", "label", pl.col("len").alias("population")),
    on=["lang", "domain", "label"],
)


def weighted_token_stats(group: list[str]) -> pl.DataFrame:
    return (
        token_weighted.group_by(group).agg(
            pl.col("sample_size").sum(),
            pl.col("population").sum(),
            *[
                (pl.col(name) * pl.col("population"))
                .sum()
                .truediv(pl.col("population").sum())
                .alias(name)
                for name in [
                    "mean_tokens",
                    "over_128",
                    "over_256",
                    "over_512",
                    "over_1024",
                    "mean_discarded_fraction_512",
                ]
            ],
        )
        if group
        else token_weighted.select(
            pl.col("sample_size").sum(),
            pl.col("population").sum(),
            *[
                (pl.col(name) * pl.col("population"))
                .sum()
                .truediv(pl.col("population").sum())
                .alias(name)
                for name in [
                    "mean_tokens",
                    "over_128",
                    "over_256",
                    "over_512",
                    "over_1024",
                    "mean_discarded_fraction_512",
                ]
            ],
        )
    )


token_overall = weighted_token_stats([])
token_by_label = weighted_token_stats(["label"])
token_by_language = weighted_token_stats(["lang"])
token_by_domain = weighted_token_stats(["domain"])
summary["tokenizer"] = {
    "model_id": "FacebookAI/xlm-roberta-base",
    "tokenizer_sha256": hashlib.file_digest(Path(tokenizer_path).open("rb"), "sha256").hexdigest(),
    "method": "up to 100 per language-domain-label group; population-weighted",
    "overall": token_overall.to_dicts(),
    "by_label": token_by_label.to_dicts(),
    "by_language": token_by_language.to_dicts(),
    "by_domain": token_by_domain.to_dicts(),
}
display(
    token_overall,
    token_by_label,
    token_by_language,
    token_by_domain,
)

sample_size,population,mean_tokens,over_128,over_256,over_512,over_1024,mean_discarded_fraction_512
i64,i64,f64,f64,f64,f64,f64,f64
2568,272352,789.728234,0.938646,0.851809,0.638011,0.242472,0.270899


label,sample_size,population,mean_tokens,over_128,over_256,over_512,over_1024,mean_discarded_fraction_512
str,i64,i64,f64,f64,f64,f64,f64,f64
"""ai""",900,121940,733.851743,0.94295,0.87053,0.669619,0.218034,0.265257
"""human""",831,66730,832.740077,0.923869,0.828138,0.652707,0.301123,0.301036
"""fusion""",837,83682,836.851856,0.944156,0.843404,0.580235,0.231313,0.25509


lang,sample_size,population,mean_tokens,over_128,over_256,over_512,over_1024,mean_discarded_fraction_512
str,i64,i64,f64,f64,f64,f64,f64,f64
"""de-DE""",468,52771,792.541854,0.982026,0.918089,0.799564,0.226605,0.309054
"""en-EN""",2100,219581,789.052048,0.92822,0.83588,0.599186,0.246285,0.26173


domain,sample_size,population,mean_tokens,over_128,over_256,over_512,over_1024,mean_discarded_fraction_512
str,i64,i64,f64,f64,f64,f64,f64,f64
"""house_of_commons""",300,31859,894.635613,0.96123,0.895795,0.719049,0.280354,0.318624
"""euro_court_cases""",300,27578,1195.805351,1.0,0.952433,0.818236,0.422134,0.427321
"""gutenberg""",468,7344,1186.909173,0.960351,0.92918,0.906068,0.569016,0.489383
"""bundestag""",300,52343,791.693961,0.983018,0.918802,0.799572,0.226967,0.308945
"""blog_authorship_corpus""",300,40547,293.421609,0.659341,0.370879,0.146842,0.031487,0.052505
"""student_essays""",300,54106,531.97931,1.0,0.933476,0.459309,0.044857,0.123533
"""cnn_news""",300,44285,852.437377,1.0,0.974552,0.805072,0.321096,0.331884
"""arxiv_papers""",300,14290,1750.652363,0.96627,0.949405,0.93254,0.803488,0.599639


**Measured observations.** In the population-weighted sample of 2,568 texts, an estimated 93.9%, 85.2%, 63.8%, and 24.2% exceed 128, 256, 512, and 1,024 tokens, respectively. The mean estimated discarded token fraction at 512 is 27.1%. Estimated rates above 512 are 58.0% for `fusion`, 67.0% for `ai`, and 65.3% for `human`. These are sample estimates, not exact population measurements.

In [7]:
display(
    sample.group_by(["lang", "label"])
    .agg(
        pl.len().alias("sample_size"),
        pl.col("tokens").median().alias("median_tokens"),
        pl.col("tokens").quantile(0.95).alias("p95_tokens"),
    )
    .sort(["lang", "label"])
)

lang,label,sample_size,median_tokens,p95_tokens
str,str,u32,f64,f64
"""de-DE""","""ai""",200,679.5,1128.0
"""de-DE""","""fusion""",137,868.0,4098.0
"""de-DE""","""human""",131,914.0,3093.0
"""en-EN""","""ai""",700,795.5,1622.0
"""en-EN""","""fusion""",700,761.0,4626.0
"""en-EN""","""human""",700,897.0,4737.0


Head-only retains the beginning; head plus tail retains two regions; sliding windows retain more context at increased cost. Chunk-based document classification requires an explicit aggregation rule and document-level supervision: no chunk inherits a verified local fusion label. I must compare policies on the same validation IDs before preferring one. The discarded fraction is a token-count proxy including special tokens, not a guarantee of exact semantic loss; estimates have sampling uncertainty.

## 06 · Duplicate and leakage investigation

**Question.** Can local splitting separate the same text or plausible variants? Exact and normalized groups are confirmed string matches under the respective definitions. I inspect within-sample candidate pairs sharing the same first or last 80 normalized characters and evaluate character similarity. This high-precision blocking misses paraphrases and short shared passages; candidates are not verified source families.

In [8]:
duplicate_stats = {}
for column in ["text", "normalized_text"]:
    duplicate_groups = (
        train.group_by(column)
        .agg(pl.len().alias("n"), pl.col("label").n_unique().alias("labels"))
        .filter(pl.col("n") > 1)
    )
    duplicate_stats[column] = {
        "groups": duplicate_groups.height,
        "rows_in_groups": int(duplicate_groups["n"].sum() or 0),
        "conflicting_label_groups": duplicate_groups.filter(pl.col("labels") > 1).height,
        "conflicting_label_rows": int(
            duplicate_groups.filter(pl.col("labels") > 1)["n"].sum() or 0
        ),
    }
blocks = defaultdict(list)
for row in sample.select("id", "label", "normalized_text").iter_rows(named=True):
    text = row["normalized_text"]
    if len(text) >= 80:
        for block in {text[:80], text[-80:]}:
            blocks[block].append(row)
pairs = {}
for block in blocks.values():
    if len(block) > 30:
        continue
    for index, first in enumerate(block):
        for second in block[index + 1 :]:
            key = tuple(sorted((str(first["id"]), str(second["id"]))))
            if key not in pairs:
                similarity = SequenceMatcher(
                    None, first["normalized_text"], second["normalized_text"], autojunk=True
                ).ratio()
                pairs[key] = (
                    similarity,
                    first["label"] != second["label"],
                    first["normalized_text"] == second["normalized_text"],
                )
near = {
    "sample_size": sample.height,
    "candidate_pairs": len(pairs),
    "threshold": 0.9,
    "near_nonidentical_pairs": sum(
        score >= 0.9 and not exact for score, cross, exact in pairs.values()
    ),
    "cross_label_near_nonidentical_pairs": sum(
        score >= 0.9 and cross and not exact for score, cross, exact in pairs.values()
    ),
    "blocked_groups_over_30_skipped": sum(len(block) > 30 for block in blocks.values()),
}
summary["duplicates"] = duplicate_stats
summary["near_duplicates"] = near
display(pl.DataFrame([{"definition": k, **v} for k, v in duplicate_stats.items()]), near)

definition,groups,rows_in_groups,conflicting_label_groups,conflicting_label_rows
str,i64,i64,i64,i64
"""text""",0,0,0,0
"""normalized_text""",1,2,0,0


{'sample_size': 2568,
 'candidate_pairs': 319,
 'threshold': 0.9,
 'near_nonidentical_pairs': 108,
 'cross_label_near_nonidentical_pairs': 5,
 'blocked_groups_over_30_skipped': 0}

**Measured observations.** Exact raw text has no duplicate groups; one normalized group has two rows with the same label. The sampled prefix/suffix blocking generated 319 pairs, of which 108 nonidentical pairs scored at least 0.90 character sequence similarity; five of these pairs have different labels. This targeted sample and blocking do not measure the prevalence of related source documents.

A 0.90 sequence similarity threshold indicates substantial character overlap, not a source-document identity. The blocking and threshold are sensitivity choices, and this sample cannot estimate overall family prevalence. Exact text groups can be kept intact in validation; related rewrites remain an unquantified leakage risk. Investigate additional candidate schemes before claiming group reconstruction.

## 07–08 · Fusion and lightweight style

**Question.** Do simple document features and variation across consecutive regions differ by class within each language? I compute interpretable proxies on the same stratified sample. Each document is split into four equal word-count regions; regional variation is measured across those regions, never treated as chunk-level truth. Short segments, topic, formatting, language and document length can affect every indicator.

In [9]:
def style_features(text: str) -> dict[str, float]:
    words = re.findall(r"\b\w+\b", text.lower(), flags=re.UNICODE)
    sentences = [part for part in re.split(r"(?<=[.!?])\s+", text) if part.strip()]
    sentence_lengths = [len(part.split()) for part in sentences]
    chunks = np.array_split(np.array(words, dtype=object), 4)
    local_ttr = [len(set(chunk)) / len(chunk) for chunk in chunks if len(chunk)]
    local_sentence = []
    local_punctuation = []
    for chunk in np.array_split(np.array(text.split(), dtype=object), 4):
        region = " ".join(chunk)
        local_sentence.append(len(chunk) / max(1, len(re.findall(r"[.!?]+", region))))
        local_punctuation.append(
            sum(not c.isalnum() and not c.isspace() for c in region) / max(1, len(region))
        )
    paragraphs = [len(part.split()) for part in re.split(r"\n\s*\n", text) if part.strip()]
    counts = Counter(words)
    return {
        "type_token_ratio": len(counts) / max(1, len(words)),
        "repeated_word_fraction": sum(count - 1 for count in counts.values()) / max(1, len(words)),
        "punctuation_fraction": sum(not c.isalnum() and not c.isspace() for c in text)
        / max(1, len(text)),
        "digit_fraction": sum(c.isdigit() for c in text) / max(1, len(text)),
        "uppercase_fraction": sum(c.isupper() for c in text) / max(1, len(text)),
        "sentence_length_sd": float(np.std(sentence_lengths)),
        "paragraph_length_sd": float(np.std(paragraphs)),
        "chunk_ttr_sd": float(np.std(local_ttr)),
        "chunk_sentence_length_sd": float(np.std(local_sentence)),
        "chunk_punctuation_sd": float(np.std(local_punctuation)),
    }


styles = pl.DataFrame([style_features(text) for text in sample["text"]]).hstack(
    sample.select("id", "label", "lang", "domain", "words")
)
style_columns = [
    name for name in styles.columns if name not in ["id", "label", "lang", "domain", "words"]
]
style_grouped = (
    styles.group_by(["lang", "label"])
    .agg(
        pl.len().alias("sample_size"),
        *[pl.col(name).median().alias(name) for name in style_columns],
    )
    .sort(["lang", "label"])
)
summary["style_by_language_label"] = style_grouped.to_dicts()
display(style_grouped)
display(
    px.box(
        styles,
        x="label",
        y="chunk_ttr_sd",
        color="label",
        facet_col="lang",
        points=False,
        color_discrete_map=COLORS,
        title="Within-document variation in regional lexical diversity",
        labels={
            "label": "Class",
            "chunk_ttr_sd": "Standard deviation across four regions",
            "lang": "Language",
        },
    )
)

lang,label,sample_size,type_token_ratio,repeated_word_fraction,punctuation_fraction,digit_fraction,uppercase_fraction,sentence_length_sd,paragraph_length_sd,chunk_ttr_sd,chunk_sentence_length_sd,chunk_punctuation_sd
str,str,u32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""de-DE""","""ai""",200,0.578581,0.421419,0.021875,0.0,0.045616,7.117705,0.0,0.029143,3.020341,0.002876
"""de-DE""","""fusion""",137,0.542282,0.457718,0.024893,0.001746,0.046033,9.409137,0.0,0.033212,3.142484,0.003828
"""de-DE""","""human""",131,0.54461,0.45539,0.02528,0.002623,0.04811,10.61111,0.0,0.032692,3.364095,0.003442
"""en-EN""","""ai""",700,0.523197,0.476803,0.026088,0.000554,0.019476,8.201875,0.0,0.029327,3.175493,0.004452
"""en-EN""","""fusion""",700,0.467791,0.532209,0.028078,0.004536,0.025453,10.862304,0.0,0.038661,3.424083,0.005966
"""en-EN""","""human""",700,0.425572,0.574428,0.028414,0.006649,0.029731,12.163419,0.0,0.035398,3.829218,0.005436


**Measured observations.** In the unweighted English sample, median regional lexical-diversity variation is 0.029 for `ai`, 0.039 for `fusion`, and 0.035 for `human`. German sample medians are 0.029, 0.033, and 0.033, respectively. This exploratory difference is confounded by document length and subgroup composition and does not locate rewritten spans. Paragraph variation is unavailable because no text contains newline characters.

These medians are unweighted within the deliberately balanced group sample and therefore describe its composition, not the full population. Lexical diversity is length-sensitive, and the regional calculations discard punctuation for lexical diversity only. They motivate controlled future experiments, not individual-text detection or claims about where rewriting occurred.

## 09 · Qualitative inspection

I inspect deterministic short prefixes across observed language and class groups. These excerpts illustrate formatting only; they cannot verify authorship or the origin of a mixed style.

The excerpt selection is illustrative, not a representative manual annotation or evidence of authorship. A separate human inspection is needed to evaluate qualitative patterns.

In [10]:
examples = (
    sample.group_by(["lang", "label"], maintain_order=True)
    .head(2)
    .select("id", "lang", "domain", "label", pl.col("text").str.slice(0, 180).alias("excerpt"))
)
display(examples)

id,lang,domain,label,excerpt
str,str,str,str,str
"""024fce""","""de-DE""","""bundestag""","""ai""","""Frau Präsidentin, verehrte Kol…"
"""029bef""","""de-DE""","""bundestag""","""ai""","""Frau Präsidentin! Liebe Kolleg…"
"""027fd1""","""de-DE""","""bundestag""","""fusion""","""Herr Präsident! Verehrte Kolle…"
"""0496be""","""de-DE""","""bundestag""","""fusion""","""Sehr geehrte Frau Präsidentin!…"
"""054884""","""de-DE""","""bundestag""","""human""","""Danke schön. – Sehr geehrte Fr…"
…,…,…,…,…
"""17891d""","""en-EN""","""arxiv_papers""","""ai""","""Let me begin by introducing th…"
"""034e9f""","""en-EN""","""arxiv_papers""","""fusion""","""Missing Section: To analyze th…"
"""0930b7""","""en-EN""","""arxiv_papers""","""fusion""","""Draft version February 12, 202…"


## 10 · Validation design

I propose a fixed 80/20 holdout with seed 2026, which provides substantial training data and a sizable validation set, stratifying by language × domain × class where counts permit. First assign all normalized exact-text duplicates to the same side; a group-aware allocation should aim for the joint proportions and document exceptions for rare groups and conflicting labels. When group reconstruction beyond exact text is unreliable, this split remains vulnerable to related rewrites: quantify sensitivity with independent candidate audits rather than claiming it is leakage-free. Freeze the ID manifest and labels before fitting any preprocessing, reuse it for all baselines, encoders and ensembles, and evaluate language, domain and length subgroups with their supports. No manifest is created here.

## 11 · Subsequent experiments

Compare word and character TF-IDF with logistic regression or a linear SVM, fitting vectorizers on the training fold. Word features capture lexical patterns; character n-grams can capture morphology and formatting.

For multilingual encoders, compare head-only and head-plus-tail token budgets, then consider sliding windows. Any 20k–50k subset should preserve subgroup coverage and come from the training fold. Further experiments can evaluate document-level chunk aggregation, stylometric heterogeneity, and multiple-instance learning without assigning labels to individual chunks.

Evaluate macro-F1, per-class precision, recall and F1, confusion matrices, and performance by language, domain, and length. No method is presumed superior before measurement.

In [11]:
METADATA_PATH.parent.mkdir(parents=True, exist_ok=True)
METADATA_PATH.write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")
{
    "metadata": str(METADATA_PATH.relative_to(ROOT)),
    "training_rows": train.height,
    "sample_rows": sample.height,
}

{'metadata': 'data/metadata/eda_summary.json',
 'training_rows': 272352,
 'sample_rows': 2568}